**This notebook was used to test out the isolation of grandparent/parent nodes with the effective isolation of also their own children nodes as well as the update and saving process of each of these nodes. In here you can test out the call the elements of the subroutines/functions inside by defining inside them on 'subroutine_key'.**

In [1]:
%reload_ext autoreload
%autoreload 2
from fparser.two import Fortran2003 as F23
from fparser.two import Fortran2008 as F28
from fparser.two.utils import walk
import os
from typing import Dict, List,Tuple,Any
from collections import deque
import yaml
import ast
import re
import itertools

In [2]:
%cd ..

/home/ssivanes/Fgpt


/data/ssivanes/fparser-venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# Processor class
from processor import Processor
from extractor import Extractor
from isolator import Isolator
processor = Processor()

INFO     Processor initialized.

In [4]:
rest_of_path = "/data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"
target_module = "hydrol" # "hydrol" explicitsnow
work = os.getenv("work")

In [5]:
isolator = Isolator(rest_of_path, target_module, work,False)

cls = Extractor(isolator.module_dir_sp, isolator.module_tree_sp)
cls.find_subroutines()

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: Isolator                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFO     Processor initialized.

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90

INFO     Successfully parsed string!

INFO     Processor initialized.

...

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/albedo_surface.f90

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow_org.f90

INFO     Found subroutine 'explicitsnow_main' in file:                                                             
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90

INFO     Backup file already exists:                                                                               
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow_org.f90

INFO     Found external subroutine 'explicitsnow_main' in file:                                                    
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90, adding to processing queue

In [6]:
cls.subroutine_keys_all

{'explicitsnow_age',
 'explicitsnow_compactn',
 'explicitsnow_compactn_up',
 'explicitsnow_drift',
 'explicitsnow_fall',
 'explicitsnow_gone',
 'explicitsnow_grain',
 'explicitsnow_icelevels',
 'explicitsnow_icemelt',
 'explicitsnow_iceprofile',
 'explicitsnow_levels',
 'explicitsnow_main',
 'explicitsnow_maxmass',
 'explicitsnow_melt_refrz',
 'explicitsnow_profile',
 'explicitsnow_subli',
 'explicitsnow_transf',
 'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_muff',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_coef_setup',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr

In [7]:
cls.subroutine_keys_ncl

{'explicitsnow_age',
 'explicitsnow_compactn',
 'explicitsnow_compactn_up',
 'explicitsnow_drift',
 'explicitsnow_fall',
 'explicitsnow_gone',
 'explicitsnow_grain',
 'explicitsnow_icelevels',
 'explicitsnow_icemelt',
 'explicitsnow_iceprofile',
 'explicitsnow_levels',
 'explicitsnow_profile',
 'explicitsnow_subli',
 'explicitsnow_transf',
 'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_flood',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag'}

In [8]:
# FInding the parents but also using the 
print(cls.subroutine_keys_all - cls.subroutine_keys_ncl)

{'hydrol_nudge_snow', 'hydrol_soil', 'hydrol_diag_soil_flux', 'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_soil_infilt', 'hydrol_root_profile', 'hydrol_nudge_mc_diag', 'hydrol_tmc_update', 'hydrol_hydraulic_arch_tuzet_resist', 'explicitsnow_maxmass', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_main', 'explicitsnow_main', 'hydrol_vegupd', 'explicitsnow_melt_refrz', 'hydrol_split_soil', 'hydrol_muff_radial_coef_setup'}


In [9]:
cls.subroutines.keys()

dict_keys(['hydrol_main', 'hydrol_tmc_update', 'hydrol_canop', 'hydrol_vegupd', 'hydrol_flood', 'hydrol_soil', 'hydrol_soil_infilt', 'hydrol_soil_smooth_under_mcr', 'hydrol_soil_smooth_over_mcs', 'hydrol_soil_smooth_over_mcs2', 'hydrol_diag_soil_flux', 'hydrol_soil_tridiag', 'hydrol_soil_coef', 'hydrol_soil_froz', 'hydrol_soil_setup', 'hydrol_split_soil', 'hydrol_diag_soil', 'hydrol_alma', 'hydrol_nudge_mc', 'hydrol_nudge_mc_diag', 'hydrol_nudge_snow', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_hydraulic_arch_tuzet_resist', 'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_muff_radial_coef_setup', 'hydrol_muff_radial_resolution', 'hydrol_root_profile', 'explicitsnow_main', 'explicitsnow_grain', 'explicitsnow_compactn', 'explicitsnow_compactn_up', 'explicitsnow_drift', 'explicitsnow_transf', 'explicitsnow_fall', 'explicitsnow_gone', 'explicitsnow_melt_refrz', 'explicitsnow_icemelt', 'explicitsnow_icelevels', 'explicitsnow_levels', 'explicitsnow_profile', 'explicitsnow_iceprofile', 'explicits

In [10]:
%reload_ext autoreload
%autoreload 2
from transformer import Transformer
from utils import identify_replace_all
import time

In [11]:
transformer = Transformer("/home/ssivanes/Fgpt/benchmark",isolator,cls,None,config_path = "/home/ssivanes/Fgpt/template.yaml")

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: Transformer                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
subroutine_key = 'hydrol_root_profile' # explicitsnow_main hydrol_vegupd hydrol_soil hydrol_flood hydrol_alma hydrol_canop hydrol_hydraulic_arch_tuzet_calc

In [13]:
isolator.parent_subroutine_call = set()
for child_procedure in ['hydrol_root_profile']:
    isolator.isolate_procedure(cls, transformer, 'hydrol_soil',child_procedure)

INFO       Call site 1: CALL hydrol_root_profile(kjpindex, altmax, sm, smw, root_profile, root_depth)

WARNING  Implicit shape detected in the declaration REAL(KIND = r_std), DIMENSION(:, :), INTENT(IN) :: altmax

INFO     Processor initialized.

INFO     Corresponding element of "altmax" is "altmax" in call statement in subroutine "hydrol_soil"!

INFO     Found explicit shape in subroutine "hydrol_soil"!

...

INFO     Benchmark files exist. Now running the unit tests ...

 --- inside the main program ---
 --- inside the declaration_initialization routine for hydrol_root_profile ---
 --- inside the read_dummy routine for hydrol_root_profile ---
 Execution time :    0.1348800000000000     


INFO     Execution completed in /home/ssivanes/Fgpt/hydrol/hydrol_root_profile

INFO     Successfully wrote code to file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90

INFO     Successfully wrote code to file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90

In [14]:
cls.loop_dict['hydrol_root_profile']

defaultdict(set, {'kjpindex': {'ji'}, 'nvm': {'jv'}})

In [15]:
isolator.working_subroutines.keys() # This has all the currently working subroutines upon the currecly working subroutines, which means 
# if we have call statement or function call inside the current working subroutines, they are added onto the working_subroutines dict. 

dict_keys(['hydrol_root_profile'])

In [16]:
print(list(cls.dec_global[subroutine_key])) # THis will now contain all the global values that will be used inside the 
# Parent subroutine and global values of the child subroutines. that we retrieve using this isolator.collect_global_vars_decl(cls.dec_global[child_subroutine_key], cls.dec_global[subroutine_key])
# which modifies direclty the cls.dec_global[subroutine_key] of the parent declarations 

['zero', 'maxaltmax', 'zdr', 'max_root_depth', 'inode', 'znh', 'iinterface', 'istruc', 'un', 'humcste', 'err_act', 'numout', 'ipslerr_p', 'plev', 'ifunc', 'min_sechiba', 'ndepths', 'nroot_prof']


In [17]:
# Most of the elements are the retrieved variables are the same for the children as for the parent. With the exception being is that all
# subroutines and functions are placed inside the global class of each node(children or parent) as well as the declaration_initalization of 
# their attributes and the main file which is a simple python script will contain all the class and method calling. This is helpful for the 
# the rest of the process on the auto differenciation process. 

In [18]:
#declaration_stmts = list(cls.dec_global[subroutine_key].values())
#ast_nodes = transformer.convert_SPECIFICATION_PART(declaration_stmts,cls_mode=True)

In [19]:
cls.loop_dict

defaultdict(<function extractor.Extractor.__init__.<locals>.<lambda>()>,
            {'hydrol_initialize': defaultdict(set, {'nslm': {'jsl'}}),
             'hydrol_main': defaultdict(set,
                         {'kjpindex': {'ji'},
                          'nvm': {'jv'},
                          'nstm': {'jst'},
                          'itopmax': {'jsl'},
                          'nslm': {'jsl'}}),
             'hydrol_init': defaultdict(set,
                         {'nslm': {'jsl'},
                          'nstm': {'jst'},
                          'kjpindex': {'ji'},
                          'nvm': {'jv'}}),
             'hydrol_tmc_update': defaultdict(set,
                         {'kjpindex': {'ji'},
                          'nvm': {'jv'},
                          'nstm': {'jst'},
                          'nslm - 1': {'jsl'},
                          'nslm': {'jsl'}}),
             'hydrol_var_init': defaultdict(set,
                         {'nslm': {'jsl'},
     

In [20]:
tree = transformer.update_global_python(subroutine_key,cls_mode = True,for_loop=True)
# THis the global module for the parent subroutine itself

╭───────────────────────────────────────────────── 🚀 Node Start ─────────────────────────────────────────────────╮
│ Entering node: Update_global_python                                                                             │
│ Function: update_global_python(subroutine_key='hydrol_root_profile', cls_mode=True, for_loop=True)              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(0 : nslm) :: zdr

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(nvm) :: max_root_depth

...

[INFO] Since argument method_name is: __init__, placing the assign statement inside of this method

[INFO] Since argument method_name is: __init__, placing the assign statement inside of this method

[INFO] Since no index is given, WILL BE USING previous known ast Assign position with this method: __init__

[INFO] Since argument method_name is: __init__, placing the assign statement inside of this method

╭─────────────────────────────────────────────── ✅ Node Complete ────────────────────────────────────────────────╮
│ Exiting node: Update_global_python                                                                              │
│ Duration: 0.14s                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
print(ast.unparse(tree)) # ['is_tuzet_hydrol_arch']

import numpy as np
import logging
from scipy.io import FortranFile
import time
import functools

class Global_module_hydrol_root_profile:

    def __init__(self):
        self.nice = np.int32(8)
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.zero = np.float64(0.0)
        self.maxaltmax = np.float64(2.0)
        self.inode = np.int32(1)
        self.iinterface = np.int32(2)
        self.istruc = np.int32(1)
        self.un = np.float64(1.0)
        self.err_act = np.int32(1)
        self.numout = np.int32(6)
        self.plev = np.int32(0)
        self.ifunc = np.int32(2)
        self.min_sechiba = np.float64(1e-08)
        self.ndepths = np.int32

In [22]:
print(transformer.dependant_variables)

{}


In [23]:
isolator.processor.reads_in_decleration_routine

[]

In [24]:
print(walk(walk(isolator.processor.reads_in_read_routine,F23.Input_Item_List),F23.Name))

[Name('humcste'), Name('max_root_depth'), Name('zdr'), Name('znh')]


In [25]:
# Where statements 
subroutine_code = """
subroutine compute_c(a, b, c)
  ! Main WHERE block
  WHERE (a > 0)               
    c = b * 2.0   

    WHERE (b < 3.0)      
      c = b + 10.0           
    ELSEWHERE (b >= 3.0)      
      c = b - 1.0            
    END WHERE                 

  ELSEWHERE (a < 0)         
    c = -b                

  ELSEWHERE              
    c = 0.0           

  END WHERE        

end subroutine compute_c
"""

subroutine_code = """
subroutine compute_c(a, b, c, n)
  implicit none
  integer, intent(in) :: n
  real, intent(in) :: a(n), b(n)
  real, intent(out) :: c(n)
  integer :: i

  ! Example loop to process in chunks or apply conditionally
  do i = 1, n
    if (mod(i, 2) == 0) then
      ! Main WHERE block for even indices
      WHERE (a > 0.0)
        c = b * 2.0

        WHERE (b < 3.0)
          c = b + 10.0
        ELSEWHERE (b >= 3.0)
          c = b - 1.0
        END WHERE

      ELSEWHERE (a < 0.0)
        c = -b

      ELSEWHERE
        c = 0.0
      END WHERE

    else
      ! For odd indices, maybe apply a different logic
      WHERE (a < 0.0)
        c = -b * 2.0
      ELSEWHERE
        c = b
      END WHERE
    end if
  end do

end subroutine compute_c
"""
sub_parser = processor.parse_fortran_string(subroutine_code)


INFO     Successfully parsed string!

In [8]:
from f2np import F2NP
f2np = F2NP(cls)

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
#_,_,subroutine_code_ast = f2np.recursive_ast(sub_code_parser)

In [18]:
#print(ast.unparse(ast.fix_missing_locations(subroutine_code_ast[0])))

In [29]:
cls.scalar_variables[subroutine_key]

[Name('zero'),
 Name('maxaltmax'),
 Name('inode'),
 Name('iinterface'),
 Name('istruc'),
 Name('un'),
 Name('err_act'),
 Name('numout'),
 Name('plev'),
 Name('ifunc'),
 Name('min_sechiba')]

In [30]:
main_tree = transformer.update_main_python(out_module=tree,subroutine_key=subroutine_key)

╭───────────────────────────────────────────────── 🚀 Node Start ─────────────────────────────────────────────────╮
│ Entering node: Update_main_python                                                                               │
│ Function: update_main_python(out_module=<ast.Module object at 0x7fd553a602e0>,                                  │
│ subroutine_key='hydrol_root_profile')                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

...

[INFO] Inserting after last assign at position 5 inside the function : main

[INFO] Inserting after last assign at position 6 inside the function : main

[INFO] Inserting after last assign at position 7 inside the function : main

[INFO] SPECIAL CASE, READ dummy method needs to be placed after all the assign statement(variables), insert 
position: 8

╭─────────────────────────────────────────────── ✅ Node Complete ────────────────────────────────────────────────╮
│ Exiting node: Update_main_python                                                                                │
│ Duration: 0.05s                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [31]:
print(ast.unparse(main_tree))

import os
import numpy as np
import logging
from scipy.io import FortranFile
from module_global import Global_module_hydrol_root_profile

def read_dummy(altmax, sm, smw):
    print(f'--- inside the read dummy routine for hydrol_root_profile ---')
    path = f'/home/ssivanes/Fgpt/benchmark/hydrol_root_profile/dummy.bin'
    ffile = FortranFile(path, 'r')
    for var in [altmax, sm, smw]:
        if isinstance(var, np.ndarray):
            arr_shape = var.shape
            if var.dtype == np.float64:
                data = ffile.read_reals(np.float64)
            elif var.dtype == np.int32 or var.dtype == np.bool:
                data = ffile.read_ints(np.int32)
            if data.size != np.prod(arr_shape):
                continue
            var[:] = data.reshape(arr_shape, order='F')

def test_hydrol_root_profile(root_profile, root_depth):
    print('--- inside the test function for hydrol_root_profile ---')
    path = f'/home/ssivanes/Fgpt/benchmark/hydrol_root_profile/output.bin'


In [32]:
#transformer.transfer_to_pyfile(main_tree,subroutine_key,python_file_type="main")

In [33]:
#transformer.transfer_to_pyfile(tree,subroutine_key)